# Krishna Menon
# BTech AI I045

### Installing all packages

In [ ]:
!pip install h2o

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.0/266.0 MB 4.2 MB/s eta 0:00:00


In [ ]:
!pip install autokeras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 15.2 MB/s eta 0:00:00


### H2O AutoML with Churn Dataset

In [ ]:
import urllib.request

# importing the dataset
url = "https://raw.githubusercontent.com/albayraktaroglu/Datasets/master/churn.csv"
output_path = "/content/churn.csv"
urllib.request.urlretrieve(url, output_path)

print(f"File downloaded and saved to: {output_path}")

File downloaded and saved to: /content/churn.csv


In [ ]:
import h2o, pandas as pd
from h2o.automl import H2OAutoML

h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.17" 2025-10-21; OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04); OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.12/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmp9q88yqua
  JVM stdout: /tmp/tmp9q88yqua/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmp9q88yqua/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,04 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,1 month and 27 days
H2O_cluster_name:,H2O_from_python_unknownUser_anjvgm
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.147 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


In [ ]:
data=pd.read_csv("churn.csv")
X=data.drop(["Churn?","Phone"], axis=1) #I dropped Phone as AutoML thinks its irrelevant
y=data["Churn?"]

In [ ]:
from sklearn.model_selection import train_test_split as tts
X_train, X_test, y_train, y_test = tts(X, y, test_size=0.33, random_state=42)

train=h2o.H2OFrame(pd.concat([X_train, y_train], axis='columns'))
test=h2o.H2OFrame(pd.concat([X_test, y_test], axis='columns'))

train.head(4)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


State,Account Length,Area Code,Int'l Plan,VMail Plan,VMail Message,Day Mins,Day Calls,Day Charge,Eve Mins,Eve Calls,Eve Charge,Night Mins,Night Calls,Night Charge,Intl Mins,Intl Calls,Intl Charge,CustServ Calls,Churn?
IN,68,415,no,no,0,222.1,107,37.76,199.4,102,16.95,162.4,107,7.31,9.4,3,2.54,2,False.
MT,131,415,no,yes,24,135.9,60,23.1,233.2,78,19.82,210.6,121,9.48,9.4,4,2.54,1,False.
NV,87,510,no,yes,39,82.6,113,14.04,224.4,63,19.07,163.6,88,7.36,9.5,1,2.57,3,False.
NJ,95,408,yes,yes,37,220.2,109,37.43,185.3,99,15.75,205.1,82,9.23,4.1,2,1.11,0,True.


In [ ]:
aml = H2OAutoML(
    max_runtime_secs=(100),
    max_models=20,
    seed=17
)

aml.train(
    x=train.columns,
    y="Churn?",
    training_frame=train
)

AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_5_AutoML_1_20260121_152543


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    47                 47                          40710                  6            6            6             36            62            54.9149

ModelMetricsBinomial: gbm
** Reported on train data. **

MSE: 0.009095169135643998
RMSE: 0.09536859617108767
LogLoss: 0.0470219821796283
Mean Per-Class Error: 0.0044048757557583525
AUC: 0.9999384046812443
AUCPR: 0.9996383436535846
Gini: 0.9998768093624886

Confusion Matrix (Act/Pred) for max f1 @ threshold = 0.18743903941804604
        False.    True.    Error    Rate
------  --------  -------  -------  ------------
False.  1905      5        0.0026   (5.0/1910.0)
True.   2         321      0.0062   (2.0/323.0)
Total   1907      326      0.0031   (7.0/2233.0)

Maximum Metrics: Maximum metrics at their respective thresholds
metric                       threshold    value     idx
---------------------------  -----------  --------  -----
max f1                       0.187439     0.989214  173
max f2                       0.154225     0.995071  178
max f0point5                 0.3042       0.9943    161
max accuracy                 0.252205     0.996865  167
max precision                0.991066     1         0
max recall                   0.154225     1         178
max specificity              0.991066     1         0
max absolute_mcc             0.187439     0.987395  173
max min_per_class_accuracy   0.160695     0.995812  177
max mean_per_class_accuracy  0.154225     0.997906  178
max tns                      0.991066     1910      0
max fns                      0.991066     322       0
max fps                      0.00327281   1910      399
max tps                      0.154225     323       178
max tnr                      0.991066     1         0
max fnr                      0.991066     0.996904  0
max fpr                      0.00327281   1         399
max tpr                      0.154225     1         178

Gains/Lift Table: Avg response rate: 14.46 %, avg score: 14.38 %
group    cumulative_data_fraction    lower_threshold    lift     cumulative_lift    response_rate    score       cumulative_response_rate    cumulative_score    capture_rate    cumulative_capture_rate    gain     cumulative_gain    kolmogorov_smirnov
-------  --------------------------  -----------------  -------  -----------------  ---------------  ----------  --------------------------  ------------------  --------------  -------------------------  -------  -----------------  --------------------
1        0.0103                      0.977937           6.91331  6.91331            1                0.982823    1                           0.982823            0.0712074       0.0712074                  591.331  591.331            0.0712074
2        0.0201523                   0.972585           6.91331  6.91331            1                0.974906    1                           0.978952            0.0681115       0.139319                   591.331  591.331            0.139319
3        0.0300045                   0.968572           6.91331  6.91331            1                0.970441    1                           0.976158            0.0681115       0.20743                    591.331  591.331            0.20743
4        0.0403045                   0.964744           6.91331  6.91331            1                0.966642    1                           0.973726            0.0712074       0.278638                   591.331  591.331            0.278638
5        0.0501567                   0.960937           6.91331  6.91331            1                0.963176    1     

In [ ]:
aml.leaderboard.head()

model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
GBM_5_AutoML_1_20260121_152543,0.905856,0.201936,0.816773,0.142664,0.227644,0.0518219
XGBoost_3_AutoML_1_20260121_152543,0.904992,0.185352,0.849245,0.141959,0.210654,0.0443752
GBM_3_AutoML_1_20260121_152543,0.902143,0.204564,0.814273,0.145237,0.230597,0.053175
GBM_2_AutoML_1_20260121_152543,0.898054,0.214312,0.798052,0.176869,0.236992,0.056165
GBM_4_AutoML_1_20260121_152543,0.896188,0.221865,0.79315,0.157905,0.240543,0.0578607
DRF_1_AutoML_1_20260121_152543,0.89472,0.429247,0.818648,0.144144,0.243189,0.059141
XGBoost_2_AutoML_1_20260121_152543,0.892981,0.216149,0.8065,0.145214,0.233271,0.0544156
XRT_1_AutoML_1_20260121_152543,0.892637,0.299478,0.79431,0.136017,0.253944,0.0644875
XGBoost_1_AutoML_1_20260121_152543,0.884188,0.239398,0.755908,0.171598,0.253598,0.0643117
GBM_1_AutoML_1_20260121_152543,0.876898,0.264055,0.672879,0.186849,0.275512,0.0759068


In [ ]:
preds = aml.leader.predict(test)
preds.head()

gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%


predict,False.,True.
False.,0.968076,0.0319242
False.,0.992778,0.00722154
True.,0.0464287,0.953571
False.,0.98483,0.01517
False.,0.983017,0.0169832
False.,0.971781,0.0282186
False.,0.98166,0.0183396
False.,0.965383,0.0346173
False.,0.951236,0.0487641
False.,0.987485,0.0125152


In [ ]:
# Saving the model
mojo_path = aml.leader.download_mojo(path="/content/", get_genmodel_jar=True)

print(f"MOJO saved to: {mojo_path}")

MOJO saved to: /content/GBM_5_AutoML_1_20260121_152543.zip


In [ ]:
performance = aml.leader.model_performance(test)
print(performance)

ModelMetricsBinomial: gbm
** Reported on test data. **

MSE: 0.0489542777009791
RMSE: 0.22125613596232557
LogLoss: 0.19338155416180858
Mean Per-Class Error: 0.15013297872340425
AUC: 0.9128257978723404
AUCPR: 0.8317707960964067
Gini: 0.8256515957446808

Confusion Matrix (Act/Pred) for max f1 @ threshold = 0.27983377721581076
        False.    True.    Error    Rate
------  --------  -------  -------  -------------
False.  928       12       0.0128   (12.0/940.0)
True.   46        114      0.2875   (46.0/160.0)
Total   974       126      0.0527   (58.0/1100.0)

Maximum Metrics: Maximum metrics at their respective thresholds
metric                       threshold    value     idx
---------------------------  -----------  --------  -----
max f1                       0.279834     0.797203  110
max f2                       0.132658     0.767327  146
max f0point5                 0.386524     0.875796  101
max accuracy                 0.386524     0.948182  101
max precision                0.9

In [ ]:
h2o.cluster().shutdown()
print("Cluster Shutdown Complete")

H2O session _sid_ab7a closed.
Cluster Shutdown Complete


### H2O AutoML with MNIST Dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset("ylecun/mnist")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mnist/train-00000-of-00001.parquet:   0%|          | 0.00/15.6M [00:00<?, ?B/s]

mnist/test-00000-of-00001.parquet:   0%|          | 0.00/2.60M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
import h2o, pandas as pd,numpy as np
from h2o.automl import H2OAutoML

h2o.init()

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.17" 2025-10-21; OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04); OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.12/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmp29z13gmy
  JVM stdout: /tmp/tmp29z13gmy/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmp29z13gmy/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,07 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,1 month and 28 days
H2O_cluster_name:,H2O_from_python_unknownUser_g0dk58
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.168 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


In [ ]:
# Flatten x_train and x_test for tabular processing
x_train_flat = x_train.reshape(x_train.shape[0], -1)
x_test_flat = x_test.reshape(x_test.shape[0], -1)

# Create Pandas DataFrames from flattened features and labels
train_df = pd.DataFrame(x_train_flat)
train_df['label'] = y_train

test_df = pd.DataFrame(x_test_flat)
test_df['label'] = y_test

# Convert Pandas DataFrames to H2O Frames
train = h2o.H2OFrame(train_df)
test = h2o.H2OFrame(test_df)

train.head(4)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0117647,0.0705882,0.0705882,0.0705882,0.494118,0.533333,0.686275,0.101961,0.65098,1,0.968627,0.498039,0,0,0,0,0,0,0,0,0,0,0,0,0.117647,0.141176,0.368627,0.603922,0.666667,0.992157,0.992157,0.992157,0.992157,0.992157,0.882353,0.67451,0.992157,0.94902,0.764706,0.25098,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.2,0.623529,0.992157,0.623529,0.196078,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.188235,0.933333,0.988235,0.988235,0.988235,0.929412,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.211765,0.890196,0.992157,0.988235,0.937255,0.913725,0.988235,0.223529,0.0235294,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.262745,0.909804,0.152941,0,0,0,0,0,0,0,0,0,0.243137,0.317647,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.470588,0.705882,0.152941,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.486275,0.992157,1,0.247059,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.376471,0.956863,0.984314,0.992157,0.243137,0,0,0,0,0,0,0,0,0,0


In [ ]:
aml = H2OAutoML(
    max_runtime_secs=(100),
    max_models=20,
    seed=17
)

aml.train(
    x=train.columns,
    y="label",
    training_frame=train
)

AutoML progress: |
17:50:05.618: _train param, Dropping bad and constant columns: [671, 111, 672, 112, 673, 476, 754, 755, 756, 757, 758, 759, 52, 53, 10, 54, 11, 55, 56, 57, 16, 17, 18, 19, 560, 0, 1, 2, 3, 168, 4, 5, 6, 644, 7, 645, 8, 9, 727, 728, 20, 729, 21, 22, 23, 24, 25, 26, 27, 28, 29, 730, 731, 699, 30, 31, 140, 141, 780, 781, 782, 783, 700, 701, 82, 83, 84, 85]

███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OXGBoostEstimator : XGBoost
Model Key: XGBoost_1_AutoML_1_20260121_174955


Model Summary: 
    number_of_trees
--  -----------------
    30

ModelMetricsRegression: xgboost
** Reported on train data. **

MSE: 0.13912646115067753
RMSE: 0.37299659670119983
MAE: 0.23726156129521686
RMSLE: 0.117856563848586
Mean Residual Deviance: 0.13912646115067753

ModelMetricsRegression: xgboost
** Reported on validation data. **

MSE: 1.092945353675575
RMSE: 1.045440267865924
MAE: 0.6309372850997099
RMSLE: NaN
Mean Residual Deviance: 1.092945353675575

Scoring History: 
    timestamp            duration          number_of_trees    training_rmse    training_mae    training_deviance    validation_rmse    validation_mae    validation_deviance
--  -------------------  ----------------  -----------------  ---------------  --------------  -------------------  -----------------  ----------------  ---------------------
    2026-01-21 17:50:05  0.272 sec         0                  4.89713          4.05655         23.9819              4.88306            4.03455           23.8443
    2026-01-21 17:50:24  19.342 sec        5                  1.17715          0.881627        1.38568              1.42254            1.02588           2.02361
    2026-01-21 17:50:42  36.995 sec        10                 0.602741         0.365808        0.363297             1.07558            0.638366          1.15688
    2026-01-21 17:50:59  53.483 sec        15                 0.491916         0.294845        0.241982             1.0473             0.617086          1.09683
    2026-01-21 17:51:13  1 min  8.124 sec  20                 0.440224         0.27099         0.193797             1.04739            0.623603          1.09702
    2026-01-21 17:51:27  1 min 22.298 sec  25                 0.408213         0.255051        0.166638             1.04737            0.628401          1.09698
    2026-01-21 17:51:36  1 min 30.633 sec  30                 0.372997         0.237262        0.139126             1.04544            0.630937          1.09295

Variable Importances: 
variable    relative_importance    scaled_importance       percentage
----------  ---------------------  ----------------------  ----------------------
409         82515.703125           1.0                     0.19158791086085758
263         27746.265625           0.3362543682499827      0.06442227193085168
403         10022.9423828125       0.12146709054431874     0.023271626115732656
325         9788.29296875          0.11862339649365983     0.022726808713439465
464         9436.833984375         0.11436409831083288     0.021910778672858202
155         8523.0966796875        0.10329060235693774     0.019789230717125274
538         7625.86865234375       0.09241718077335659     0.017706014592017604
239         7003.16845703125       0.08487073601520922     0.01626020700637728
407         5663.14892578125       0.06863116608486453     0.013148901730143738
431         5237.390625            0.06347144151539337     0.012160360879261322
---         ---                    ---                     ---
582         1.9957313537597656     2.418607947552123e-05   4.633760438629779e-06
77          1.9832004308700562     2.403421840647445e-05   4.604665693670009e-06
66          1.5912028551101685     1.9283636869696352e-05  3.694511701664531e-06
70          1.5149058103561401     1.835900020219503e-05   3.517362494232608e-06
648         1.4556668996810913     1.7641089447858855e-05  3.3798194726247975e-06
535         1.4220912456512451     1.7234189273003847e-05  3.3018623181954092e-06
361         1.3158429861068726     1.594657666691091e-05   3.055171308995959e-06
172         0.9445236921310425     1.1446593270861649e-05  2.1930288912383338e-06
72          0.1194305419921875     1.4473674400043173e-06  2.7729810407845477e-07
771         0.09766717255115509    1.1836192246123466e-06  2.2676713449822763e-07
[506 rows x 4 columns]


[tips]
Use `model.explain()` to inspect the model.
--
Use

In [ ]:
preds = aml.leader.predict(test)
preds.head()

xgboost prediction progress: |███████████████████████████████████████████████████| (done) 100%


predict
6.87177
1.26443
0.928313
-0.0308252
3.92726
0.987548
3.90959
7.42296
7.56036
8.77537


In [ ]:
# Saving the model
mojo_path = aml.leader.download_mojo(path="/content/", get_genmodel_jar=True)

print(f"MOJO saved to: {mojo_path}")

MOJO saved to: /content/XGBoost_1_AutoML_1_20260121_174955.zip


In [ ]:
performance = aml.leader.model_performance(test)
print(performance)

ModelMetricsRegression: xgboost
** Reported on test data. **

MSE: 1.1251517889410465
RMSE: 1.0607317233594207
MAE: 0.6236284499179455
RMSLE: NaN
Mean Residual Deviance: 1.1251517889410465


In [ ]:
h2o.cluster().shutdown()
print("Cluster Shutdown Complete")

H2O session _sid_b31c closed.
Cluster Shutdown Complete


### AutoKeras with MNIST Dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset("ylecun/mnist")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mnist/train-00000-of-00001.parquet:   0%|          | 0.00/15.6M [00:00<?, ?B/s]

mnist/test-00000-of-00001.parquet:   0%|          | 0.00/2.60M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/60000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
import autokeras as ak,numpy as np
# Initialize the ImageClassifier
clf = ak.ImageClassifier(
    max_trials=1, # Reduced for faster training
    objective="val_accuracy"
)

print("AutoKeras ImageClassifier initialized successfully.")

AutoKeras ImageClassifier initialized successfully.


In [ ]:
x_train = np.array([np.array(image) for image in ds['train']['image']], dtype='float32')
y_train = np.array(ds['train']['label'])
x_test = np.array([np.array(image) for image in ds['test']['image']], dtype='float32')
y_test = np.array(ds['test']['label'])

x_train = x_train / 255.0
x_test = x_test / 255.0

print("MNIST data prepared:")
print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_test shape: {y_test.shape}")

MNIST data prepared:
x_train shape: (60000, 28, 28)
y_train shape: (60000,)
x_test shape: (10000, 28, 28)
y_test shape: (10000,)


In [ ]:
clf.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=10
)

Trial 1 Complete [00h 01m 21s]
val_accuracy: 0.9919000267982483

Best val_accuracy So Far: 0.9919000267982483
Total elapsed time: 00h 01m 21s
Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9106 - loss: 0.2883 - val_accuracy: 0.9832 - val_loss: 0.0461
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9773 - loss: 0.0733 - val_accuracy: 0.9872 - val_loss: 0.0392
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9820 - loss: 0.0588 - val_accuracy: 0.9867 - val_loss: 0.0388
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9851 - loss: 0.0484 - val_accuracy: 0.9886 - val_loss: 0.0338
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9868 - loss: 0.0399 - val_accuracy: 0.9888 - val_loss: 0.0356
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9873 - loss: 0.0405 - val_accuracy: 0.9893 - val_loss: 0.0332
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9885 - lo

In [ ]:
# Get the best model
best_model = clf.export_model()
print("Best model exported successfully.")
print(f"Model type: {type(best_model)}")
best_model.summary()

Best model exported successfully.
Model type: <class 'keras.src.models.functional.Functional'>


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cast_to_float32 (CastToFloat32) │ (None, 28, 28)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ expand_last_dim (ExpandLastDim) │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalization (Normalization)   │ (None, 28, 28, 1)      │             3 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        92,170 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classification_head_1 (Softmax) │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,989 (433.55 KB)

 Trainable params: 110,986 (433.54 KB)

 Non-trainable params: 3 (16.00 B)

In [ ]:
# Make predictions on test data
predictions = clf.predict(x_test)
print("Predictions shape:", predictions.shape)
print("First 10 predictions:")
print(predictions[:10].flatten())

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Predictions shape: (10000, 1)
First 10 predictions:
['7' '2' '1' '0' '4' '1' '4' '9' '5' '9']


In [ ]:
# Save the model
model_path = "autokeras_mnist_model.keras" # Added .keras extension
best_model.save(model_path)
print(f"Model saved to: {model_path}")

Model saved to: autokeras_mnist_model.keras


### AutoKeras with Iris Dataset

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import autokeras as ak

# Load the Iris dataset
iris = load_iris()
X, y = iris.data, iris.target

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")
print(f"Number of classes: {len(iris.target_names)}")
print(f"Class names: {iris.target_names}")

Training data shape: (105, 4)
Testing data shape: (45, 4)
Number of classes: 3
Class names: ['setosa' 'versicolor' 'virginica']


In [ ]:
# Initialize AutoKeras StructuredDataClassifier
clf_iris = ak.StructuredDataClassifier(
    max_trials=10,
    objective="val_accuracy"
)

print("AutoKeras StructuredDataClassifier initialized successfully.")

AutoKeras StructuredDataClassifier initialized successfully.


In [ ]:
# Train the model
clf_iris.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test)
)

Trial 10 Complete [00h 00m 09s]
val_accuracy: 1.0

Best val_accuracy So Far: 1.0
Total elapsed time: 00h 05m 16s


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
# Get the best model and make predictions
best_model_iris = clf_iris.export_model()
predictions_iris = clf_iris.predict(X_test)

print("Iris predictions shape:", predictions_iris.shape)
print("First 10 predictions:")
print(predictions_iris[:10].flatten())

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 824ms/step
Iris predictions shape: (45, 1)
First 10 predictions:
['1' '0' '2' '1' '1' '0' '1' '2' '1' '1']


In [ ]:
# Save the Iris model
iris_model_path = "autokeras_iris_model.keras"
best_model_iris.save(iris_model_path)
print(f"Iris model saved to: {iris_model_path}")

Iris model saved to: autokeras_iris_model.keras


### Zero-Shot Learning Lab

#### Phase 1: Unstructured Prompt
Model Used: Gemini 3 Thinking\
Time Taken: 3 min\
Confidence Level: 2

```I want you to use zero-shot meta-learning approach and write the code to create a model where the model can classify an unseen image without task-specific training. Do this in python. Give me the code and avoid text explaing it. You can add comments into the code.```

In [ ]:
import torch
from PIL import Image
import requests
from transformers import CLIPProcessor, CLIPModel

# ==============================================================================
# Zero-Shot Image Classification using CLIP
#
# Approach:
# Instead of training a meta-learning loop from scratch (which is complex and
# data-intensive), we utilize CLIP (Contrastive Language-Image Pre-training).
# CLIP is a multimodal model pre-trained on massive amounts of image-text pairs.
# It learns to align images and text descriptions in a shared embedding space.
#
# To perform zero-shot classification on a new task:
# 1. We provide an unseen image.
# 2. We provide a list of potential text labels (classes) for that image.
# 3. The model encodes the image and all text labels into the same vector space.
# 4. It calculates the cosine similarity between the image vector and each
#    text label vector.
# 5. The label with the highest similarity score is the prediction.
#
# This requires NO task-specific training on the newly provided labels.
# ==============================================================================

# --- 1. Setup Model and Processor ---
# Using the base CLIP model from OpenAI via Hugging Face transformers.
model_id = "openai/clip-vit-base-patch32"
print(f"Loading pre-trained model: {model_id}...")

# The model manages the underlying neural networks (Vision Transformer & Text Transformer)
model = CLIPModel.from_pretrained(model_id)

# The processor handles necessary preprocessing: resizing/normalizing images
# and tokenizing the input text labels.
processor = CLIPProcessor.from_pretrained(model_id)


# --- 2. Define the "Unseen" Task (Input Data) ---

# Define the potential classes for the classification task.
# The model has not been specifically fine-tuned on these exact label strings as a closed set.
candidate_labels = [
    "a photo of a beagle",
    "a photo of a golden retriever",
    "a photo of a german shepherd",
    "a photo of a persian cat",
    "a photo of a toaster"
]

# Load an unseen image to classify.
# We will load a Golden Retriever image from Wikimedia.
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c0/Golden_Retriever_with_tennis_ball.jpg/640px-Golden_Retriever_with_tennis_ball.jpg"

try:
    # Use timeout and check for status codes to ensure 'image' is created
    response = requests.get(image_url, stream=True, timeout=10)
    response.raise_for_status()
    image = Image.open(response.raw).convert("RGB")
except Exception as e:
    print(f"Error loading image: {e}")
    # Fallback: Create a blank image so the rest of the code doesn't crash with NameError
    image = Image.new('RGB', (224, 224), color='gray')


# --- 3. Preprocessing and Inference (The Zero-Shot Step) ---
print("Running zero-shot classification...")

# Process the image and text labels together.
# The processor prepares the inputs into tensors expected by the model.
# return_tensors="pt" asks for PyTorch tensors.
inputs = processor(
    text=candidate_labels,
    images=image,
    return_tensors="pt",
    padding=True
)

# Perform forward pass. Disable gradient calculation as we are only doing inference.
with torch.no_grad():
    outputs = model(**inputs)

# The model outputs `logits_per_image`, which represents the similarity scores
# between the provided image(s) and the provided text labels.
# Shape will be (batch_size_images, num_labels) -> (1, 5) in this example.
logits_per_image = outputs.logits_per_image

# Apply Softmax over the labels dimension (dim=1) to convert logits to probabilities that sum to 1.
probs = logits_per_image.softmax(dim=1)

# Move to CPU and convert to numpy for easy reading.
prob_scores = probs.cpu().numpy()[0]


# --- 4. Display Results ---
print("\n--- Classification Results ---")

# Combine labels with scores and sort descending by score.
results = sorted(zip(candidate_labels, prob_scores), key=lambda x: x[1], reverse=True)

for label, score in results:
    print(f"{label:<30} | Confidence: {score:.4f}")

predicted_label = results[0][0]
print(f"\nFinal Prediction: {predicted_label}")

Loading pre-trained model: openai/clip-vit-base-patch32...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Error loading image: 403 Client Error: Forbidden for url: https://upload.wikimedia.org/wikipedia/commons/thumb/c/c0/Golden_Retriever_with_tennis_ball.jpg/640px-Golden_Retriever_with_tennis_ball.jpg
Running zero-shot classification...

--- Classification Results ---
a photo of a german shepherd   | Confidence: 0.3950
a photo of a golden retriever  | Confidence: 0.2388
a photo of a persian cat       | Confidence: 0.2142
a photo of a beagle            | Confidence: 0.1248
a photo of a toaster           | Confidence: 0.0273

Final Prediction: a photo of a german shepherd


#### Phase 2: Structured Prompt
Model Used: Gemini 3 Thinking
Time Taken: 3 min\
Confidence Level: 5
```You are an expert in machine learning and computer vision.
Write Python code that performs image identification using a zero-shot meta-learning approach.
The code should classify an unseen image without any task-specific training.
Use a pre-trained model capable of zero-shot inference.
Do not fine-tune the model.
Include all necessary imports.
Add comments explaining each step.
Output a complete Python script.
```

In [ ]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-oo51zw55
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-oo51zw55
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=15672e563c1fde2acc8f511a5783dafc1d9995b56fddfeff201da2f98d0f2fdf
  Stored in directory: /tmp/pip-ephem-wheel-cache-49sz3z1o/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [ ]:
import torch
import clip
from PIL import Image
import requests
from io import BytesIO

def perform_zero_shot_classification():
    # 1. Setup device (Use GPU if available, otherwise CPU)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    # 2. Load the pre-trained CLIP model and its specific preprocessing pipeline
    # We use "ViT-B/32" (Vision Transformer), a balanced model for speed/accuracy.
    # No fine-tuning is performed; we load the weights as-is.
    model, preprocess = clip.load("ViT-B/32", device=device)

    # 3. Load an unseen image from a URL
    # For this example, we use a photo of a Golden Retriever.
    url = "https://images.unsplash.com/photo-1552053831-71594a27632d?auto=format&fit=crop&w=640&q=80"
    response = requests.get(url)
    image = Image.open(BytesIO(response.content)).convert("RGB")

    # 4. Define candidate labels (The "Zero-Shot" part)
    # You can change these to anything; the model has never seen this specific
    # list of classes before, but it understands the concepts.
    class_names = ["a golden retriever", "a tesla car", "a slice of pepperoni pizza", "a sunset over mountains"]

    # 5. Preprocess the image and tokenize the text
    # Image is resized, cropped, and normalized to the model's expected format.
    image_input = preprocess(image).unsqueeze(0).to(device)
    # Text is converted into numerical tokens the model can understand.
    text_inputs = clip.tokenize(class_names).to(device)

    # 6. Perform Inference (Zero-Shot Matching)
    with torch.no_grad():
        # Calculate image and text features (embeddings)
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_inputs)

        # Pick the top most similar label
        # We compute the cosine similarity between the image and all labels.
        # The 'logits_per_image' represents the raw scores.
        logits_per_image, logits_per_text = model(image_input, text_inputs)

        # Convert raw scores (logits) into probabilities using Softmax
        probs = logits_per_image.softmax(dim=-1).cpu().numpy()

    # 7. Output results
    print("Zero-Shot Classification Results:")
    for label, prob in zip(class_names, probs[0]):
        print(f"{label}: {prob:.4f}")

    # Identify the highest probability label
    best_match_idx = probs.argmax()
    print(f"\nFinal Prediction: {class_names[best_match_idx]}")

if __name__ == "__main__":
    perform_zero_shot_classification()

100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 148MiB/s]


Zero-Shot Classification Results:
a golden retriever: 1.0000
a tesla car: 0.0000
a slice of pepperoni pizza: 0.0000
a sunset over mountains: 0.0000

Final Prediction: a golden retriever


### Conclusion of Experiments

This notebook explored different AutoML frameworks and a Zero-Shot Learning approach for various machine learning tasks.

1.  **H2O AutoML:** Demonstrated strong performance in predicting customer churn from a tabular dataset. However, when applied to the MNIST image dataset, it treated the task as a regression problem due to the numerical nature of the labels, highlighting the importance of understanding how AutoML frameworks interpret data types.

2.  **AutoKeras:** Proved to be an effective and user-friendly tool for both image classification (MNIST dataset) and structured data classification (Iris dataset). It automatically searched for and optimized neural network architectures, achieving high accuracy without manual model design.

3.  **Zero-Shot Learning (CLIP):** Illustrated a powerful capability to classify unseen images without any task-specific training. By leveraging a pre-trained model that understands the relationship between images and text, it successfully identified objects based solely on textual descriptions, showcasing remarkable generalization ability.

Overall, the experiments demonstrated the diverse strengths of different AutoML solutions and the emerging power of zero-shot learning for tasks requiring generalization to novel classes.